---
title: Examples
description: Practical examples of using the Sampler primitive in Qiskit Runtime.
---


# Sampler examples

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

Generate entire error-mitigated quasi-probability distributions sampled from quantum circuit outputs. Leverage Sampler’s capabilities for search and classification algorithms like Grover’s and QVSM.

## Run a single experiment

Use Sampler to return the measurement outcome as bitstrings or counts of a single circuit.

In [5]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import random_hermitian
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

n_qubits = 127

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)

sampler = Sampler(backend)
job = sampler.run([isa_circuit])
result = job.result()

# Get results for the first (and only) PUB
pub_result = result[0]

print(f" > First ten results: {pub_result.data.meas.get_bitstrings()[:10]}")

 > First ten results: ['0101110000110001001111000101001111000110110100011000100101011101110011010010010101000110000111101010101000001010000100100000100', '0100010101111101010000100010011100110001010000011000000010001100010111000011001010000100100000100000000010000000010010101011110', '1101010111111111100010000011101010101010100100011001000000001001110010001000000010000010000101000111000100010010000001111000010', '1001110001100001001101111010111100000100010110010001001100111000110010111000001010001000000000000000100101101001110010101000110', '0001000000011011000011000111001000000000100110110011111110110100110000101010100010000010101011011000101011101000100000110000011', '1011100010011111010000001110110000111101000001110010011001100011111010001100100000110001000010001010110011100010000111000111010', '1101110000011000001011011000001111001110010111111111100100010001110100000010000001011000110000000011010011110100101001101000010', '01101000001100110000110010001101101100010001001000011110100

## Run multiple experiments in a single job

Use Sampler to return the measurement outcome as bitstrings or counts of multiple circuits in one job.

In [6]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.quantum_info import random_hermitian
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

n_qubits = 127

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

rng = np.random.default_rng()
mats = [np.real(random_hermitian(n_qubits, seed=rng)) for _ in range(3)]
circuits = [iqp(mat) for mat in mats]
for circuit in circuits:
    circuit.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuits = pm.run(circuits)

sampler = Sampler(mode=backend)
job = sampler.run(isa_circuits)
result = job.result()

for idx, pub_result in enumerate(result):
    print(
        f" > First five results for pub {idx}: {pub_result.data.meas.get_bitstrings()[:5]}"
    )

 > First ten results for pub 0: ['1000000101000100010111001010101010000001001010101011011001011110001000000110110101010000010000000000110001001000011111110000001', '1111101011011011110001011000001100001101100001000101111011101110000101111010001011111010001001000010111001110111000010001011010', '1100100101110010000110101011110010111001101010001101100010110100110110000110110010001110001000001010011100001000011011000111010', '1100010010000100100010110100011010011001010101101101101001100001001110011001011011111100011100100001000101010000111101110001101', '0011101011101100010011111001001110000101100110000110000001111000011010011110000110100000110011011000000010110001010000111000100', '0110101101110000010110100100010011000100100010000010010010110001111111110000101011000100010000000100100100110011010111101110111', '1101011000111100011000010110000010001100101011000001110010110001111101010101011110110010000100011101000001010110010101000000100', '0000101010010100000010111110111000001011000000001

## Run parameterized circuits

Run several experiments in a single job, leveraging parameter values to increase circuit reusability.

In [7]:
import numpy as np
from qiskit.circuit.library import real_amplitudes
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

n_qubits = 127

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

# Step 1: Map classical inputs to a quantum problem
circuit = real_amplitudes(num_qubits=n_qubits, reps=2)
circuit.measure_all()

# Define three sets of parameters for the circuit
rng = np.random.default_rng(1234)
parameter_values = [
    rng.uniform(-np.pi, np.pi, size=circuit.num_parameters) for _ in range(3)
]

# Step 2: Optimize problem for quantum execution.

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)

# Step 3: Execute using Qiskit primitives.
sampler = Sampler(backend)
job = sampler.run([(isa_circuit, parameter_values)])
result = job.result()
# Get results for the first (and only) PUB
pub_result = result[0]
# Get counts from the classical register "meas".
print(
    f" >> First five results for the meas output register: {pub_result.data.meas.get_bitstrings()[:5]}"
)

 >> First ten results for the meas output register: ['1100011011100001011000001001000001111110000001011100011110011100111110000111000100011100001111100010010111110001001111011000101', '1100011101010101010000100110110110010001100101011101001011101010111110000111110100000011111010101101011101101101001111011110011', '0000000011000011001101001000111110001100010010011011001111000101000000001111111101101011100111010110111101010111011001010001011', '0101010001101110100010001100111001011101101100001000100001011101110100001000011011001011110101000110010001001010011011100011101', '0110101110000010110000001000010101100010010001001001101000010100110001011111110001000001100110010001011111001010011001001000101', '0111011111110111010111100110101000010100101000001010001001011111010010100111110110000011100001100000110000111000011011100000000', '0110100111001000100100110110010001011110000000110111000011110000100111001000100110011100100001100000101111111100010111100111001', '01011011110101100000000010000

## Use batches and advanced options

Explore the batch [execution mode](/docs/guides/execution-modes) and advanced options to optimize circuit performance on QPUs.

In [1]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Batch, SamplerV2 as Sampler
from qiskit_ibm_runtime import QiskitRuntimeService

n_qubits = 127

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=n_qubits
)

rng = np.random.default_rng(1234)
mat = np.real(random_hermitian(n_qubits, seed=rng))
circuit = iqp(mat)
circuit.measure_all()
mat = np.real(random_hermitian(n_qubits, seed=rng))
another_circuit = iqp(mat)
another_circuit.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(circuit)
another_isa_circuit = pm.run(another_circuit)

# The context manager automatically closes the batch.
with Batch(backend=backend) as batch:
    sampler = Sampler(mode=batch)
    job = sampler.run([isa_circuit])
    another_job = sampler.run([another_isa_circuit])
    result = job.result()
    another_result = another_job.result()

# first job

print(
    f" > The first five measurement results of job 1: {result[0].data.meas.get_bitstrings()[:5]}"
)

 > The first five measurement results of job 1: ['1001111000111001100010010111000111101101000000101000010010101001110000000010001110010001011000100100000101010010000001001000010', '0000010000001101010100011001001011011010000110000100011000000000011000010111101100010101011100100101000110011000110000000000011', '0001100011110010110100110010111001001110101100100010011001100100111000110011000100000000100001001001100100101010110010000111101', '1000100111000111010011111010010111011001100000001001101010100001101010010110100100001010000000101110100010000000100110001100000', '1011011111101100000001001010111100001001111010000000001011001000011010000101110000101010101011000110110011010011011000010000001']


In [2]:
# second job
print(
    " > The first five measurement results of job 2:",
    another_result[0].data.meas.get_bitstrings()[:5],
)

 > The first five measurement results of job 2: ['1100111000010110001000101110100001011010101100101001111000100100010101111111001000000000000000110000001100110001001000000010000', '0011010001111000001011011010011000110010101111001100000000011110011011110010010011010000000010010011001011001110010001000100100', '0101001001101010011010011000001001010000001111001100001001011100110001001001001110100001101000000101000001000011000000000110100', '0100010010100000101000001001100010000110100111010000101010010110111111110010000011001110000001100000001011000000000100000000001', '1101000000001110110101011000011111111101011101100010000001011010010001110100001010001010010110100010000010100011000000010100100']


## Next steps

<Admonition type="tip" title="Recommendations">
    - [Specify advanced runtime options](runtime-options-overview).
    - Practice with primitives by working through the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
    - Learn how to transpile locally in the [Transpile](/docs/guides/transpile/) section.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
    - Understand the [Job limits](/docs/guides/job-limits) when sending a job to an IBM&reg; QPU.
</Admonition>